In [1]:
from src.scenarios.wind_production import WindProductionGenerator

model = WindProductionGenerator(model='conditional_nvp')

In [14]:
import pandas as pd
import numpy as np

# Forecast dataset
df_forecasts = pd.read_csv("../data_samples/energinet_forecasts_5min_offshore_wind_2025-2026.csv")
df_forecasts["Minutes5UTC"] = pd.to_datetime(df_forecasts["Minutes5UTC"], utc=True)
df_forecasts["TimestampUTC"] = pd.to_datetime(df_forecasts["TimestampUTC"], utc=True)
df_forecasts["Minutes5DK"] = pd.to_datetime(df_forecasts["Minutes5DK"], utc=False)
df_forecasts["TimestampDK"] = pd.to_datetime(df_forecasts["TimestampDK"], utc=False)

# Capacity dataset
df_capacity = pd.read_csv("../data_samples/energinet_capacity_per_municipality_DK2_2025-2026.csv")
capacity = df_capacity['OffshoreWindCapacity'].values[-1]

ow_forecast = df_forecasts[['Minutes5UTC', 'ForecastDayAhead']].set_index('Minutes5UTC')

# Normalise to capacity
ow_forecast = ow_forecast / capacity

ow_forecast = ow_forecast.resample('15min').mean()

In [15]:
# get one day of forecast data (96 timesteps of 15 minutes)
X_fcst_test = []
dates_test = []
for day in pd.date_range("2025-01-01", "2025-01-31", freq="D"):
    day_str = day.strftime("%Y-%m-%d")
    if day_str in ow_forecast.index.strftime("%Y-%m-%d").unique():
        X_fcst_test.append(ow_forecast.loc[day_str].values.flatten())
        dates_test.append(day_str)
X_fcst_test = np.array(X_fcst_test)

In [16]:
X_fcst_test

array([[0.83094007, 0.82774906, 0.82423894, ..., 0.85487268, 0.85295807,
        0.85231987],
       [0.85168166, 0.85168166, 0.83445019, ..., 0.69021635, 0.68734444,
        0.68447253],
       [0.68223882, 0.6793669 , 0.67553769, ..., 0.76648159, 0.75818495,
        0.74829281],
       ...,
       [0.14359563, 0.14933946, 0.15412598, ..., 0.7246793 , 0.71861638,
        0.71223435],
       [0.70808603, 0.70266131, 0.69723658, ..., 0.43972174, 0.43748803,
        0.43525432],
       [0.43429702, 0.43270151, 0.4288723 , ..., 0.32803625, 0.32324973,
        0.317825  ]], shape=(31, 96))

In [17]:
model.generate(n_scenarios=100, wind_forecast=X_fcst_test[0])

array([[0.88650644, 0.87188244, 0.98504055, ..., 0.8652551 , 0.8462273 ,
        0.8530946 ],
       [0.7922521 , 0.79620254, 0.840801  , ..., 0.8496354 , 0.85105866,
        0.8300361 ],
       [0.8298891 , 0.8972131 , 0.91579866, ..., 0.9266113 , 0.8826188 ,
        0.8021553 ],
       ...,
       [0.88748163, 0.87948376, 0.8425385 , ..., 0.81731653, 0.8877089 ,
        0.90165025],
       [0.7764276 , 0.74643135, 0.7702198 , ..., 0.88149154, 0.85475475,
        0.79135007],
       [0.8634695 , 0.9034573 , 0.9090848 , ..., 0.8638013 , 0.88138235,
        0.9224644 ]], shape=(100, 96), dtype=float32)